# m02 lab · Volume conduction and Poisson's equation

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

m01 put sources in an infinite, uniform conductor. Here you solve for the potential when the conductor is not uniform:
you derive Poisson's equation from current conservation, solve it on a grid, add an interface between two
conductivities and check the grid against the method of images, and use reciprocity to see which tissue an electrode
is sensitive to.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Poisson's equation for a Gaussian source; a finite-difference solver and its convergence | eq-m02-poisson, eq-m02-fd |
| B | A flat interface: images, continuity and the electrode-potential factor | eq-m02-interface, eq-m02-images |
| C | Two media on a grid: where the interface sits matters | eq-m02-fd, eq-m02-images |
| D | Reciprocity and lead fields: what an electrode and a bipole see | eq-m02-reciprocity |

Relative units, distances in mm; conductivity ratios are parameters, not tissue values. Fields are treated as
quasi-static (Plonsey and Heppner 1967). Each part ends with **checks**. No downloads; an Octave copy of parts A–D is
in `labs/octave/m02_lab.m`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import sources as S, volume as V

## Part A · Poisson's equation on a grid

Current is conserved: what a source injects per unit volume, $I_v$, leaves as divergence of the current density
$\mathbf J = -\sigma\nabla\phi$. So $\nabla\cdot(\sigma\nabla\phi) = -I_v$ (eq-m02-poisson). A Gaussian blob of total
current $I$ in two dimensions has an exact potential, $-\frac{I}{2\pi\sigma}\left(\ln r + \tfrac12 E_1(r^2/2s^2)\right)$.
Solve the same problem with the 5-point finite-difference scheme (eq-m02-fd) and halve the spacing twice.

In [ ]:
errs = []
for n in (40, 80, 160):
    X, Y, h = V.grid(n, 10.0)
    exact = V.gaussian_potential_2d(np.hypot(X, Y))
    phi = V.solve_poisson(np.ones_like(X), V.gaussian_source_2d(X, Y), h, exact)
    errs.append(np.max(np.abs(phi - exact)))
    print(f"{n:4d} nodes per side, spacing {h:.3f} mm: largest error {errs[-1]:.2e}")
orders = [np.log2(errs[i] / errs[i + 1]) for i in range(2)]
print("observed order of accuracy:", np.round(orders, 2))
mid = n // 2
plt.plot(X[mid], exact[mid], "k-", label="exact")
plt.plot(X[mid][::6], phi[mid][::6], "o", ms=3, label="finite differences")
plt.xlabel("x (mm)"); plt.ylabel("potential (relative)"); plt.legend(); plt.show()

In [ ]:
# Checks
assert all(abs(o - 2) < 0.2 for o in orders)          # second-order accurate
assert errs[-1] < 2e-4

## Part B · A flat interface and the method of images

At an interface the potential is continuous and so is the normal current: $\phi_1 = \phi_2$ and
$\sigma_1\,\partial\phi_1/\partial n = \sigma_2\,\partial\phi_2/\partial n$ (eq-m02-interface). For a point source in
medium 1 a depth $d$ below a flat interface, both conditions are met if medium 1 sees the source plus a mirror-image source $kI$ at height $+d$, with
$k = (\sigma_1 - \sigma_2)/(\sigma_1 + \sigma_2)$, and medium 2 sees the source alone with strength $(1 + k)I$, which
gives $\phi_2 = I/(2\pi(\sigma_1 + \sigma_2) r)$ (eq-m02-images). Check continuity
numerically, then see how the potential at an electrode just above the interface depends on the conductivity
ratio.

In [ ]:
s1, d = 1.0, 2.0
jumps = []
for s2 in (0.25, 4.0):
    eps = 1e-6
    below = np.array([[0.7, 0.3, -eps]]); above = np.array([[0.7, 0.3, eps]])
    pb, pa = float(V.image_potential_3d(1, s1, s2, d, below)[0]), float(V.image_potential_3d(1, s1, s2, d, above)[0])
    # normal current, by one-sided differences
    hz = 1e-4
    jb = -s1 * (float(V.image_potential_3d(1, s1, s2, d, [[0.7, 0.3, -eps]])[0]) - float(V.image_potential_3d(1, s1, s2, d, [[0.7, 0.3, -eps - hz]])[0])) / hz
    ja = -s2 * (float(V.image_potential_3d(1, s1, s2, d, [[0.7, 0.3, eps + hz]])[0]) - float(V.image_potential_3d(1, s1, s2, d, [[0.7, 0.3, eps]])[0])) / hz
    jumps.append((s2, abs(pa - pb) / abs(pb), abs(ja - jb) / abs(jb)))
    print(f"sigma2 = {s2}: potential mismatch {jumps[-1][1]:.1e}, normal-current mismatch {jumps[-1][2]:.1e}")

ratios = np.logspace(-2, 1, 200)
factor = 2 / (1 + ratios)                 # electrode in medium 2, against the same source with medium 1 everywhere
plt.semilogx(ratios, factor)
plt.xlabel("sigma2 / sigma1"); plt.ylabel("potential relative to medium 1 everywhere"); plt.show()
print(f"insulating medium 2: x{2 / (1 + 0):.1f}; equal media: x{2 / 2:.1f}; medium 2 ten times more conductive: x{2 / 11:.2f}")

In [ ]:
# Checks
for s2, dphi, dj in jumps:
    assert dphi < 1e-4 and dj < 1e-2
pt = np.array([[0.4, -0.2, 1.0]])
assert np.isclose(float(V.image_potential_3d(1, 1, 1e-12, 2.0, pt)[0]), 2 * float(S.point_potential(1, 1, pt, (0, 0, -2.0))[0]), rtol=1e-6)

An insulating boundary doubles the potential; a better-conducting medium on the electrode's side shunts current
and lowers it. The influence of the blood mass inside the heart on surface leads was analyzed in exactly these terms
by Brody (1956).

## Part C · Two media on a grid

Now solve the two-media problem on a grid, with conductivities on nodes and harmonic means on the faces between them
(eq-m02-fd), and compare with the 2-D image solution. Try it twice: with the interface on a row of nodes (odd number of
nodes) and halfway between rows (even number).

In [ ]:
def two_media_error(n, s2=3.0):
    X, Y, h = V.grid(n, 10.0)
    sig = np.where(Y <= 0, 1.0, s2)
    exact = V.image_potential_2d(1.0, 1.0, s2, 2.0, X, Y)
    phi = V.solve_poisson(sig, V.gaussian_source_2d(X, Y + 2.0, 1.0, 0.3), h, exact)
    far = np.hypot(X, Y + 2.0) > 2.0
    return np.max(np.abs(phi - exact)[far]), X, Y, phi, exact

on_faces = [two_media_error(n)[0] for n in (80, 160, 320)]
on_nodes = [two_media_error(n)[0] for n in (81, 161, 321)]
order_faces = [np.log2(on_faces[i] / on_faces[i + 1]) for i in range(2)]
order_nodes = [np.log2(on_nodes[i] / on_nodes[i + 1]) for i in range(2)]
print("interface on cell faces: errors", np.array(on_faces).round(7), "order", np.round(order_faces, 2))
print("interface on a row of nodes: errors", np.array(on_nodes).round(7), "order", np.round(order_nodes, 2))
err, X, Y, phi, exact = two_media_error(80)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].contourf(X, Y, phi, 30); ax[0].axhline(0, color="k"); ax[0].set_title("finite differences")
ax[1].contourf(X, Y, exact, 30); ax[1].axhline(0, color="k"); ax[1].set_title("images")
plt.show()

In [ ]:
# Checks
assert all(o > 1.8 for o in order_faces)
assert all(o < 1.3 for o in order_nodes)       # an interface on nodes is shifted by half a cell: first order

Where the interface sits in the grid decides the accuracy: a half-cell shift is a first-order error, whatever
else the scheme does. Real meshes of blood and tissue (m32, m33) face the same problem.

## Part D · Reciprocity and lead fields

Helmholtz reciprocity: the potential that a dipole $\mathbf p$ at $\mathbf r_0$ produces at an electrode equals
$\mathbf p\cdot\nabla\psi(\mathbf r_0)$, where $\psi$ is the potential made by unit current injected at the electrode
(eq-m02-reciprocity). So one solve per electrode gives its sensitivity to every source: its **lead field**. Check it
across the interface, then compare how fast a single electrode's and a bipole's sensitivity fall with depth.

In [ ]:
p = np.array([0.3, -0.5, 1.0])
pairs = []
for e in ([1.0, 0.5, 1.5], [0.2, -0.3, 0.4], [-2.0, 1.0, 3.0]):
    for s2 in (0.25, 4.0):
        direct = float(V.image_dipole_potential_3d(p, 1.0, s2, 2.0, [e])[0])
        recip = V.reciprocity_two_media(p, 1.0, s2, 2.0, e)
        pairs.append((direct, recip))
print("direct vs reciprocity (first three):", [(round(a, 6), round(b, 6)) for a, b in pairs[:3]])

depth = np.linspace(0.2, 8, 100)
pts = np.stack([0 * depth, 0 * depth, -depth], -1)
uni, bi = V.bipole_lead_magnitude(pts, 2.0, 1.0)
plt.semilogy(depth, uni / uni[0], label="unipolar electrode, 1 mm above tissue")
plt.semilogy(depth, bi / bi[0], label="bipole, 2 mm spacing")
plt.xlabel("depth of the source (mm)"); plt.ylabel("sensitivity (relative to 0.2 mm)"); plt.legend(); plt.show()
drop_uni, drop_bi = uni[-1] / uni[0], bi[-1] / bi[0]
print(f"sensitivity at 8 mm against 0.2 mm: unipolar {drop_uni:.1e}, bipole {drop_bi:.1e}")

In [ ]:
# Checks
assert all(abs(a - b) < 1e-5 * abs(a) for a, b in pairs)
assert drop_bi < drop_uni

A bipole's lead field falls off faster with depth than a single electrode's: that is the far-field rejection of
m03, seen from the electrode's side.

## Where to go next

- In part B, put the electrode in medium 1 instead. How does the factor change with the conductivity ratio?
- In part C, make medium 2 a hundred times more conductive. Does the solver still converge at second order?
- In part D, rotate the dipole parallel to the interface. Which electrode sees it better?
- m03 turns these potentials into electrograms; m07 asks how to run the problem backwards.